In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score , mean_absolute_error , mean_squared_error


In [2]:
df = pd.read_csv("insurance.csv")
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [3]:
# First 5 rows
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
# Last 5 rows
df.tail()

,age,sex,bmi,children,smoker,region,charges
1333,50,male,30.97,3,no,northwest,10600.5483
1334,18,female,31.92,0,no,northeast,2205.9808
1335,18,female,36.85,0,no,southeast,1629.8335
1336,21,female,25.80,0,no,southwest,2007.9450
1337,61,female,29.07,0,yes,northwest,29141.3603


In [5]:
print("Shape of the Data: ")
print(df.shape)

Shape of the Data: 
(1338, 7)


In [6]:
print("Summary of the dataset: ")
print(df.info())

Summary of the dataset: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 57.6+ KB
None


In [7]:
# Datatypes of Each Column
print("Datatypes of Each Column :")
print(df.dtypes)

Datatypes of Each Column :
age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object


In [8]:
# Numerical Columns Statistical Summary
print("Statistic Summary for Numerical Columns:")
print(df.describe())

Statistic Summary for Numerical Columns:
               age          bmi     children       charges
count  1338.000000  1338.000000  1338.000000   1338.000000
mean     39.207025    30.663397     1.094918  13270.422265
std      14.049960     6.098187     1.205493  12110.011237
min      18.000000    15.960000     0.000000   1121.873900
25%      27.000000    26.296250     0.000000   4740.287150
50%      39.000000    30.400000     1.000000   9382.033000
75%      51.000000    34.693750     2.000000  16639.912515
max      64.000000    53.130000     5.000000  63770.428010


In [9]:
# Missing Values check
missing_values = df.isnull().sum()
print("Missing Values for Each Column")
print(missing_values)

Missing Values for Each Column
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64


In [10]:
# Duplicate values Check
duplicates = df.duplicated().sum()
print("Duplicate Row Count: ")
print(duplicates)

Duplicate Row Count: 
1


In [11]:
# Dropping duplicate Rows 
df = df.drop_duplicates()
# shape after dropping duplicate rows
df.shape
print("Shape after removing duplicate rows - ", df.shape)

Shape after removing duplicate rows -  (1337, 7)


In [12]:
# create new datafile for input variables
X = df.drop("charges", axis=1)

# create new datafile for target variable
y = df["charges"]

In [13]:
# Encoding using One-Hot Encoding
cat_cols = ['sex','smoker','region']

encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(X[cat_cols])

encoded_df = pd.DataFrame( encoded, columns=encoder.get_feature_names_out(cat_cols), index=X.index)

# adding encoded columns to df
X = pd.concat([X, encoded_df], axis=1)
X.head()

,age,sex,bmi,children,smoker,region,sex_female,sex_male,smoker_no,smoker_yes,region_northeast,region_northwest,region_southeast,region_southwest
0,19,female,27.900,0,yes,southwest,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,18,male,33.770,1,no,southeast,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
2,28,male,33.000,3,no,southeast,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
3,33,male,22.705,0,no,northwest,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
4,32,male,28.880,0,no,northwest,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0


In [32]:
# Removing categorial columns for model training
X_model = X.drop(columns=cat_cols)

In [33]:
X_train , X_test, y_train, y_test = train_test_split(X_model, y , test_size = 0.2, random_state = 42)

In [34]:
# train baseline model
baseline_model = RandomForestRegressor(random_state = 42)

baseline_model.fit(X_train , y_train)

baseline_pred = baseline_model.predict(X_test)

In [35]:
print("Baseline R2:" , r2_score(y_test, baseline_pred))
print("Baseline MAE:",mean_absolute_error(y_test, baseline_pred))
print("Baseline RMSE", np.sqrt(mean_squared_error(y_test, baseline_pred)))

Baseline R2: 0.8824310851066982
Baseline MAE: 2563.595462848507
Baseline RMSE 4648.010440433211


In [36]:
#Hyperparameter Grid
param_grid = {
    "n_estimators" : [100 , 200] ,
    "max_depth" : [None , 10 , 20] ,
    "min_samples_split" : [2 , 5] ,
    "max_features" : [0.8 , 1.0]
}

In [37]:
# Grid search optimization
model = RandomForestRegressor(random_state = 42)

grid = GridSearchCV(
    estimator = model , 
    param_grid = param_grid , 
    cv = 5 ,
    scoring = "r2" ,
    n_jobs = -1
)

grid.fit(X_train , y_train)

,estimator,RandomForestR...ndom_state=42)
,param_grid,"{'max_depth': [None, 10, ...], 'max_features': [0.8, 1.0], 'min_samples_split': [2, 5], 'n_estimators': [100, 200]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [38]:
print("Best Parameters:" , grid.best_params_)
print("Best CV Score :" , grid.best_score_)

Best Parameters: {'max_depth': 10, 'max_features': 0.8, 'min_samples_split': 5, 'n_estimators': 100}
Best CV Score : 0.8337643301985018


In [39]:
best_model_ = grid.best_estimator_

predictions = best_model_.predict(X_test)

In [40]:
print("Optimized R2:" , r2_score(y_test, predictions))
print("Optimized  MAE:",mean_absolute_error(y_test, predictions))
print("Optimized  RMSE", np.sqrt(mean_squared_error(y_test, predictions)))

Optimized R2: 0.8927354839175995
Optimized  MAE: 2477.002213849314
Optimized  RMSE 4439.651482557513


In [42]:
# Model Compariosn
print("Baseline R2:", round(r2_score(y_test, baseline_pred), 2))
print("Optimized R2:", round(r2_score(y_test, predictions), 2))

Baseline R2: 0.88
Optimized R2: 0.89
